# Procedural exclusions

Pablo Rogers

In [ ]:
#| label: setup-01
#| include: false

# Load required packages for data manipulation and table formatting
library(tidyverse)
library(gt)

## Data reading and preparation

The `data.csv` file contains the raw survey responses (separator `;`). In this step, dropout records are removed, variables for the analytical flow are selected, and the total response time (`TIME`) is derived.

In [ ]:
#| label: data-reading

# Read raw dataset
raw_data <- read_csv2(
    "../../Data/InputData/data.csv",
    show_col_types = FALSE
)

ℹ Using "','" as decimal and "'.'" as grouping mark. Use `read_delim()` for more control.

Rows: 1,326
Columns: 33
$ ID       <dbl> 10441016017, 10440981694, 10440601239, 10440562981, 104403585…
$ CREATED  <chr> "28/12/2018 12:00", "28/12/2018 11:25", "28/12/2018 03:23", "…
$ MODIFIED <chr> "28/12/2018 12:20", "28/12/2018 11:47", "28/12/2018 03:39", "…
$ CQ1      <dbl> 1, 1, 1, 1, 1, 1, 1, 1, NA, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ CQ2      <dbl> 1, 1, 1, 1, 1, 1, 1, 1, NA, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ CQ3      <dbl> 1, 1, 1, 1, 1, 1, 1, 1, NA, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ Q1       <dbl> 5, 3, 1, 4, 3, 4, 3, 4, 2, 5, 4, 3, 4, 2, 4, 5, 4, 5, 4, 4, 4…
$ Q2       <dbl> 5, 4, 1, 4, 4, 4, 4, 4, 2, 5, 4, NA, 4, 1, 4, 5, 4, 4, 4, 4, …
$ Q3       <dbl> 1, 2, 3, 2, 1, 1, 1, 1, 4, 2, 3, 2, 2, 3, 3, 1, 2, 2, 2, 3, 1…
$ Q4       <dbl> 1, 2, 2, 1, 1, 1, 2, 1, 4, 1, 3, 2, 2, 2, 2, 2, 2, 1, 2, 1, 1…
$ Q5       <dbl> 3, 3, 1, 3, 3, 3, 3, 3, 1, 4, 3, 4, 4, 1, 3, 4, 3, 5, 3, 4, 3…
$ Q6       <dbl> 4, 4, 1, 3, 5, 4, 3, 4, 4, 4, 5, 3, 3, 4, 4, 5, 3, 5, 4, 4, 3…
$ Q7       <dbl>

## Procedural exclusions (Level 1)

-   `CQ == 1`: the respondent passed the quality control item.
-   `CQ == 0`: the respondent failed the quality control item.

In [ ]:
#| label: level-1

# Evaluate procedural exclusion criteria (time, quality control fails, and missing values)
level1_data <- imported_data |>
    mutate(
        qc_fails = rowSums(across(CQ1:CQ3, \(x) x != 1), na.rm = TRUE),
        na_count = rowSums(across(Q1:Q26, is.na)),
        excl_time = TIME < 208,
        excl_qc = qc_fails >= 2,
        excl_na = na_count > 5,
        level1_excluded = excl_time | excl_qc | excl_na
    )

# Build a log of all excluded respondents and their specific exclusion criterion
exclusion_log <- bind_rows(
    prepared_data |>
        filter(dropout) |>
        transmute(
            ID,
            criterion = "dropout",
            value     = NA_real_,
            detail    = "No valid responses in Q1-Q26"
        ),
    level1_data |>
        filter(excl_time) |>
        transmute(
            ID,
            criterion = "fast_time",
            value     = TIME,
            detail    = paste0("TIME = ", TIME, " s")
        ),
    level1_data |>
        filter(excl_qc) |>
        transmute(
            ID,
            criterion = "qc_fails",
            value     = qc_fails,
            detail    = paste0("QC fails = ", qc_fails)
        ),
    level1_data |>
        filter(excl_na) |>
        transmute(
            ID,
            criterion = "excessive_missings",
            value     = as.numeric(na_count),
            detail    = paste0("Missings in Q1-Q26 = ", na_count)
        )
) |>
    distinct(ID, criterion, .keep_all = TRUE)

# Export the exclusion log
write_csv2(exclusion_log, "../../Output/DataAppendixOutput/exclusion_log.csv")

# Filter eligible respondents and invert reverse-scored scale items
eligible_data <- level1_data |>
    filter(!level1_excluded) |>
    select(ID, CQ1, CQ2, CQ3, TIME, Q1:Q26) |>
    # Inversion of reverse-scored items (6 - value)
    mutate(across(c(Q3, Q4, Q26), ~ 6 - .x))

### Level 1 Summary

In [ ]:
#| label: level-1-summary

# Generate a summary table of procedural exclusions during Level 1
tibble(
    Step = c(
        "Raw data",
        "Dropouts removed",
        "Excluded by time (< 208 s)",
        "Excluded by QC (≥ 2 fails)",
        "Excluded by missings (> 5 in Q1-Q26)",
        "Eligible for Level 2",
        "Remaining NAs in Q1-Q26"
    ),
    N = c(
        nrow(raw_data),
        sum(prepared_data$dropout, na.rm = TRUE),
        sum(level1_data$excl_time, na.rm = TRUE),
        sum(level1_data$excl_qc, na.rm = TRUE),
        sum(level1_data$excl_na, na.rm = TRUE),
        nrow(eligible_data),
        sum(is.na(eligible_data |> select(Q1:Q26)))
    )
) |>
    knitr::kable(caption = "Procedural exclusions summary")

  Step                                         N
  --------------------------------------- ------
  Raw data                                  1546
  Dropouts removed                           220
  Excluded by time (\< 208 s)                  0
  Excluded by QC (≥ 2 fails)                  28
  Excluded by missings (\> 5 in Q1-Q26)        3
  Eligible for Level 2                      1295
  Remaining NAs in Q1-Q26                     90

  : Procedural exclusions summary


In [ ]:
#| label: level-1-export
#| include: false

# Save eligible respondents data for Level 2 post-hoc indicators analysis
dir.create("../../Data/IntermediateData", recursive = TRUE, showWarnings = FALSE)
write_csv2(eligible_data, "../../Data/IntermediateData/whoqol_imported.csv")